# **Library Setup**

In [7]:
import torch
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device}")
if device == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using Device: cuda
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU


# **Test Character-Level Tokenizers**

In [8]:
# CELL 2: The Dataset
text = """Hello, Helia! Welcome to the DAI project. 
This is a simple character-level tokenizer test to see how text becomes numbers."""

print("Length of dataset in characters:", len(text))
print("First 30 characters:", text[:30])

Length of dataset in characters: 123
First 30 characters: Hello, Helia! Welcome to the D


In [9]:
# CELL 3: Extracting Unique Characters
chars = sorted(list(set(text)))
vocab_size = len(chars)

print("Vocabulary (all unique characters):", "".join(chars))
print("Vocabulary Size:", vocab_size)

Vocabulary (all unique characters): 
 !,-.ADHITWabcehijklmnoprstuvwxz
Vocabulary Size: 33


In [10]:
# CELL 4: Creating the Encoding & Decoding Dictionaries
# stoi = String TO Integer
stoi = {ch: i for i, ch in enumerate(chars)}

# itos = Integer TO String
itos = {i: ch for i, ch in enumerate(chars)}

print("Number ID for 'H':", stoi.get('H'))
print("Character for ID 10:", itos.get(10))

Number ID for 'H': 8
Character for ID 10: T


In [19]:
# CELL 5: The Encoder and Decoder
def encode(s):
    # Takes a string, outputs a list of integers
    return [stoi[c] for c in s]
    # If a character isn't in 'stoi', it defaults to 0 instead of crashing
    # return [stoi.get(c, 0) for c in s]

def decode(l):
    # Takes a list of integers, outputs a string
    return ''.join([itos[i] for i in l])
    # Same here, defaults to an empty string if ID is unknown
    # return ''.join([itos.get(i, '') for i in l])

# Let's test it out
test_string = "DAI project"
encoded_test = encode(test_string)
decoded_test = decode(encoded_test)

print(f"Original: {test_string}")
print(f"Encoded:  {encoded_test}")
print(f"Decoded:  {decoded_test}")

Original: DAI project
Encoded:  [7, 6, 9, 1, 24, 25, 23, 18, 15, 14, 27]
Decoded:  DAI project


In [13]:
# CELL 6: Converting to PyTorch Tensors
# We encode the entire dataset and wrap it in a tensor
data = torch.tensor(encode(text), dtype=torch.long)

# Move the data to your GPU
data = data.to(device)

print("Data tensor shape:", data.shape)
print("Data tensor type:", data.dtype)
print(f"Data is currently on: {data.device}")
print("First 15 encoded tokens:\n", data[:15])

Data tensor shape: torch.Size([123])
Data tensor type: torch.int64
Data is currently on: cuda:0
First 15 encoded tokens:
 tensor([ 8, 15, 20, 20, 23,  3,  1,  8, 15, 20, 17, 12,  2,  1, 11],
       device='cuda:0')


In [20]:
# CELL 7: Splitting the Data (Train vs Validation)
# We will use 90% of the data to train, and keep 10% to validate
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print(f"Total tokens in dataset: {len(data)}")
print(f"Tokens for Training: {len(train_data)}")
print(f"Tokens for Validation: {len(val_data)}")

Total tokens in dataset: 123
Tokens for Training: 110
Tokens for Validation: 13


In [21]:
# CELL 8: Data Loader
torch.manual_seed(1337) # Setting a seed ensures our random numbers are predictable for debugging

batch_size = 32 # How many independent sequences process in parallel (Your RTX 4060 can easily handle 32+)
block_size = 8  # Maximum context length: how many characters it looks at to predict the next

def get_batch(split):
    # Choose training or validation set
    data_split = train_data if split == 'train' else val_data
    
    # Generate random starting points in the dataset
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    
    # x is the input context, y is the target (the exact same sequence shifted by 1 character)
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    
    # Move the batches to the GPU
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print("Input shape (Batch, Time):", xb.shape)
print("Target shape (Batch, Time):", yb.shape)

Input shape (Batch, Time): torch.Size([32, 8])
Target shape (Batch, Time): torch.Size([32, 8])


In [59]:
# CELL 9: The Baseline Neural Network
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # An embedding table that maps each character ID directly to the next predicted ID's probability
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (batch_size, block_size) tensors
        logits = self.token_embedding_table(idx) # Outputs shape: (Batch, Time, Channels/Vocab)
        
        if targets is None:
            loss = None
        else:
            # PyTorch's cross_entropy expects different tensor shapes, so we reshape them
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            # Calculate how far off the predictions are from the true targets
            loss = F.cross_entropy(logits, targets)
            
        return logits, loss

model = BigramLanguageModel(vocab_size)
m = model.to(device) # Move model to RTX 4060
logits, loss = m(xb, yb)
print(f"Initial, completely random loss: {loss.item():.4f}")

Initial, completely random loss: 3.7214


In [60]:
# CELL 10: The Training Loop
# AdamW is the standard optimizer used for almost all modern LLMs
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

training_steps = 1000

for step in range(training_steps):
    # 1. Grab a batch of data
    xb, yb = get_batch('train')
    
    # 2. Forward pass: evaluate the loss
    logits, loss = m(xb, yb)
    
    # 3. Backward pass: clear old gradients, calculate new ones
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    
    # 4. Update the model weights
    optimizer.step()
    
    # Print progress
    if step % 100 == 0:
        print(f"Step {step:4d} | Loss: {loss.item():.4f}")

print(f"Final trained loss: {loss.item():.4f}")

Step    0 | Loss: 3.8057
Step  100 | Loss: 3.6657
Step  200 | Loss: 3.5344
Step  300 | Loss: 3.2624
Step  400 | Loss: 3.1847
Step  500 | Loss: 3.0519
Step  600 | Loss: 2.9495
Step  700 | Loss: 2.8866
Step  800 | Loss: 2.7533
Step  900 | Loss: 2.7055
Final trained loss: 2.4574


In [64]:
# CELL 11: Text Generation
# We start with a blank context (a 1x1 tensor holding the ID for a space or newline)
context = torch.zeros((1, 1), dtype=torch.long, device=device)

def generate_text(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        # 1. Get the predictions from the model
        logits, _ = model(idx)
        
        # 2. Focus only on the very last step in the sequence
        logits = logits[:, -1, :] 
        
        # 3. Apply softmax to convert raw numbers into probabilities (0.0 to 1.0)
        probs = F.softmax(logits, dim=-1)
        
        # 4. Sample from the distribution (adds a bit of creative randomness)
        idx_next = torch.multinomial(probs, num_samples=1)
        
        # 5. Append the new character to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)
        
    return idx

# Generate 200 characters and decode them back to English!
generated_ids = generate_text(m, context, max_new_tokens=200)
generated_text = decode(generated_ids[0].tolist())

print("--- GENERATED TEXT ---")
print(generated_text)

--- GENERATED TEXT ---

, hom.nmelxmDuIrh AHHIbeilrabknaara Hw
xtivphpmuuzbcw
vrzwrhpojabelc!Ipose.zHj.rDuxarolow
kTDnn
owbDkelra wbe bp,TmAhpokeelcW.rujealowbxDI twrb-lcAjAu 
kelken-xallWD.lWuA
zlctIi-
TbaIe Hz! HIhecnWvlsj
